# Comparative Analysis of Machine Learning Models

## Workflow:

1. Library Imports
2. Data Loading & Preprocessing
3. Model Definition
4. Evaluation (Cross-Validation)
5. Visualization (ROC Curves & Confusion Matrices)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix, accuracy_score, 
                             precision_score, recall_score, f1_score)
from sklearn.linear_model import LogisticRegressionCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


In [ ]:
try:
    data_ALS_original = pd.read_csv("data.ttest_ALS_original.csv")
    data_ALS_validation = pd.read_csv("data.ttest_ALS_validation.csv")
    print("ALS data successfully uploaded.")
    print(f"Original Cohort: {data_ALS_original.shape[0]} samples.")
    print(f"Validación Cohort: {data_ALS_validation.shape[0]} samples.")
except FileNotFoundError:
    print("Error.")

In [ ]:
# Convert the 'diagnosis' column to binary: 0 = Control, 1 = Disease
data_ALS_original['class'] = data_ALS_original['diagnosis'].apply(lambda x: 1 if x == 'ALS' else 0)
data_ALS_validation['class'] = data_ALS_validation['diagnosis'].apply(lambda x: 1 if x == 'ALS' else 0)

print(data_ALS_original['class'].value_counts())

print(data_ALS_validation['class'].value_counts())

In [ ]:
# List of genes for ALS
genes_ALS = ['LILRB2','TLN1','ITGAM','C5AR1','RPL15','FOS','EIF4A2','NKTR','TPT1',
             'RBM25','RPL22','TAX1BP1','RPS6KB1','IL18','PTPRC','PTEN','CTSS',
             'TNFSF13B','TLR4','CSF1R']

# Training data (original cohort)
X_train = data_ALS_original[genes_ALS]
y_train = data_ALS_original['class']

# Test data (external validation cohort)
X_val = data_ALS_validation[genes_ALS]
y_val = data_ALS_validation['class']

# Cross-validation
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Model 1: LASSO (Logistic Regression with L1)
pipe_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegressionCV(
        Cs=np.logspace(-3, 3, 50), 
        cv=5, # *Internal* CV to find the best C
        penalty='l1', 
        solver='liblinear',
        scoring='roc_auc', 
        random_state=42, 
        max_iter=5000, 
        class_weight='balanced'
    ))
])

# Model 2: SVM 
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(probability=True, 
                  random_state=42, 
                  class_weight='balanced'))
])

# Model 3: Random Forest
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42, 
                                     class_weight='balanced'))
])

models = {
    'LASSO': pipe_lasso,
    'SVM_base': pipe_svm,
    'RF_base': pipe_rf
}

print(f"Models: {list(models.keys())}")

In [ ]:

#HYPERPARAMETER OPTIMISATION

#SVM
# We will optimise 'C' (cost/penalty) and 'gamma' (RBF kernel).
# We will use the 'rbf' (Gaussian) kernel to test non-linearity.
param_grid_svm = {
    'model__kernel': ['rbf'], # RBF (non-lineal)
    'model__C': [0.1, 1, 10, 50, 100], # Different penalty values
    'model__gamma': ['scale', 'auto', 0.01, 0.1, 1] # Different kernel widths
}

#Random Forest
# Let's optimise 'n_estimators' (number of trees) and 'max_depth' (depth)
param_grid_rf = {
    'model__n_estimators': [50, 100, 200], # Number of trees in the forest
    'model__max_depth': [None, 5, 10, 20], # Maximum depth of each tree
    'model__min_samples_leaf': [1, 2, 4] # Minimum samples on a leaf node
}

# Configure GridSearchCV for SVM
# We use 'cv=cv_strategy' to ensure it uses the same 10-fold split
grid_svm = GridSearchCV(
    estimator=pipe_svm,        # Pipeline
    param_grid=param_grid_svm, # Grid
    scoring='roc_auc',         # Target metric
    cv=cv_strategy,            # CV strategy
    n_jobs=-1                  # all
)

# Configuring GridSearchCV for Random Forest
grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    scoring='roc_auc',
    cv=cv_strategy,
    n_jobs=-1
)

print("Optimising SVM...")
grid_svm.fit(X_train, y_train)

print("Optimising Random Forest...")
grid_rf.fit(X_train, y_train)

print("\n--- ¡Optimising completed! ---")
print(f"Best parameters for SVM: {grid_svm.best_params_}")
print(f"Best AUC (CV) for SVM: {grid_svm.best_score_:.4f}")

print(f"\nBest parameters for Random Forest: {grid_rf.best_params_}")
print(f"Best AUC (CV) for Random Forest: {grid_rf.best_score_:.4f}")


models = {
    'LASSO': pipe_lasso,
    'SVM': grid_svm.best_estimator_, # .best_estimator_ It is the pipeline already trained with the best parameters.
    'Random Forest': grid_rf.best_estimator_
}

print(f"Models ready for the evaluation: {list(models.keys())}")

In [ ]:

# EVALUATION AND COMPARISON

def get_metrics(y_true, y_pred, y_prob):
    metrics = {
        'AUC': roc_auc_score(y_true, y_prob),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'Specificity': recall_score(y_true, y_pred, pos_label=0, zero_division=0), 
        'F1-Score': f1_score(y_true, y_pred)
    }
    return metrics

def run_cv_evaluation(models_dict, X, y, cv_strategy):
    results = {}
    
    print("\nInitiating Evaluation in Original Cohort.")
    
    for name, model in models_dict.items():
        if name == 'LASSO':
            model.fit(X, y)
        
        # cross_val_predict 
        y_pred = cross_val_predict(model, X, y, cv=cv_strategy, n_jobs=-1, method='predict')
        y_prob = cross_val_predict(model, X, y, cv=cv_strategy, n_jobs=-1, method='predict_proba')[:, 1]
        
        # Metrics
        cv_metrics = get_metrics(y, y_pred, y_prob)
        cv_metrics['Model'] = name
        cv_metrics['Cohort'] = 'Original (CV)'
        
        results[name] = cv_metrics
    
    print("Evaluation (CV) completed.")
    return results

def evaluate_on_validation(models_dict, X_train, y_train, X_val, y_val):
    results = {}
    
    print("\nStarting Evaluation in Validation Cohort.")

    for name, model in models_dict.items():
        
        model.fit(X_train, y_train)
        
        y_pred_val = model.predict(X_val)
        y_prob_val = model.predict_proba(X_val)[:, 1]
        
        val_metrics = get_metrics(y_val, y_pred_val, y_prob_val)
        val_metrics['Model'] = name
        val_metrics['Cohort'] = 'Validation'
        
        results[name] = val_metrics
        
    print("Evaluation (Validation) completed.")
    return results

# Results

cv_results = run_cv_evaluation(models, X_train, y_train, cv_strategy)
val_results = evaluate_on_validation(models, X_train, y_train, X_val, y_val)

df_cv = pd.DataFrame(cv_results.values())
df_val = pd.DataFrame(val_results.values())
df_results = pd.concat([df_cv, df_val])

cols = ['Model', 'Cohort', 'AUC', 'Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score']
df_results = df_results[cols].set_index(['Model', 'Cohort']).sort_index()

print("\n--- Results ---")
print(df_results)

In [ ]:
print("\nROC graphs and confusion matrices.")

# ROC graphs Original Cohort
plt.figure(figsize=(10, 8))
plt.title('ROC Curve - Original Cohort', fontsize=16)

for name, model in models.items():
    if name == 'LASSO':
        model.fit(X_train, y_train) 
    
    y_prob = cross_val_predict(model, X_train, y_train, cv=cv_strategy, n_jobs=-1, method='predict_proba')[:, 1]
    fpr, tpr, _ = roc_curve(y_train, y_prob)
    auc = roc_auc_score(y_train, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--') 
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(loc='lower right', fontsize=12)
plt.grid(True)
plt.savefig('ROC_Cohorte_Original_CV.png', dpi=600, bbox_inches='tight')
plt.show()

# Confusion matrices Original Cohort
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion matrices - Original cohort', fontsize=16)

for i, (name, model) in enumerate(models.items()):
    ax = axes[i]
    if name == 'LASSO':
        model.fit(X_train, y_train)
        
    y_pred = cross_val_predict(model, X_train, y_train, cv=cv_strategy, n_jobs=-1, method='predict')
    cm = confusion_matrix(y_train, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(name, fontsize=14)
    ax.set_xlabel('Prediction', fontsize=12)
    ax.set_ylabel('Actual Value', fontsize=12)
    ax.set_xticklabels(['Control', 'ALS'])
    ax.set_yticklabels(['Control', 'ALS'])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('ConfusionMatrix_Cohorte_Original_CV.png', dpi=600, bbox_inches='tight')
plt.show()


# ROC graphs Validation Cohort
plt.figure(figsize=(10, 8))
plt.title('ROC Curve - Validation Cohort', fontsize=16)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob_val = model.predict_proba(X_val)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_val, y_prob_val)
    auc = roc_auc_score(y_val, y_prob_val)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--') 
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(loc='lower right', fontsize=12)
plt.grid(True)
plt.savefig('ROC_Cohorte_Validacion.png', dpi=600, bbox_inches='tight')
plt.show()

# Confusion matrices Validation Cohort
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion matrices - Validation cohort', fontsize=16)

for i, (name, model) in enumerate(models.items()):
    ax = axes[i]
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    cm = confusion_matrix(y_val, y_pred_val)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(name, fontsize=14)
    ax.set_xlabel('Prediction', fontsize=12)
    ax.set_ylabel('Actual Value', fontsize=12)
    ax.set_xticklabels(['Control', 'ALS'])
    ax.set_yticklabels(['Control', 'ALS'])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('ConfusionMatrix_Cohorte_Validacion.png', dpi=600, bbox_inches='tight')
plt.show()
